In [1]:
import os
if os.getenv("CUDA_VISIBLE_DEVICES") is None:
    gpu_num = 0 # Use "" to use the CPU
    os.environ["CUDA_VISIBLE_DEVICES"] = f"{gpu_num}"
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
# tahtan
# Import Sionna
import sys
sys.path.append('../')
import sionna

# try:
#     import sionna
# except ImportError as e:
#     # Install Sionna if package is not already installed
#     import os
#     os.system("pip install sionna")
#     import sionna

import tensorflow as tf
# Configure the notebook to use only a single GPU and allocate only as much memory as needed
# For more details, see https://www.tensorflow.org/guide/gpu
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    try:
        tf.config.experimental.set_memory_growth(gpus[0], True)
    except RuntimeError as e:
        print(e)
# Avoid warnings from TensorFlow
tf.get_logger().setLevel('ERROR')

sionna.config.seed = 42 # Set seed for reproducible results

# Load the required Sionna components
from sionna.nr import PUSCHConfig, PUSCHTransmitter, PUSCHReceiver, CarrierConfig, PUSCHDMRSConfig,\
                        TBConfig, PUSCHPilotPattern, TBEncoder, PUSCHPrecoder, LayerMapper, LayerDemapper, check_pusch_configs,\
                        TBDecoder, PUSCHLSChannelEstimator
from sionna.nr.utils import generate_prng_seq
from sionna.channel import AWGN, RayleighBlockFading, OFDMChannel, TimeChannel, time_lag_discrete_time_channel
from sionna.channel.utils import * 
from sionna.channel.tr38901 import Antenna, AntennaArray, UMi, UMa, RMa, TDL, CDL
from sionna.channel import gen_single_sector_topology as gen_topology
from sionna.utils import compute_ber, ebnodb2no, sim_ber, array_to_hash, create_timestamped_folders, b2b, f2f, BinarySource
from sionna.ofdm import KBestDetector, LinearDetector, MaximumLikelihoodDetector,\
        LSChannelEstimator, LMMSEEqualizer, RemoveNulledSubcarriers, ResourceGridDemapper,\
        ResourceGrid, ResourceGridMapper, OFDMModulator
from sionna.mimo import StreamManagement
from sionna.mapping import Mapper, Demapper


In [2]:
%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import time
from datetime import datetime, timedelta
# from bs4 import BeautifulSoup
import pickle
from collections import namedtuple
import json
from tqdm.notebook import tqdm
import itertools
import io
import h5py

## A Hello World Example

Let us start with a simple "Hello, World!" example in which we will simulate PUSCH transmissions from a single transmitter to a single receiver over an AWGN channel.

In [5]:
def load_pickle(filename):
    """Saves data to a pickle file."""
    with open(filename, "rb") as f:
        return pickle.load(f)

# Function to read a single sample from HDF5
def load_hdf5(filename, name):
    with h5py.File(filename, "r") as f:
        grp = f[f"{name}"]
        b = grp["b"][:]
        c = grp["c"][:]
        y = grp["y"][:]
    return b, c, y

def data_loader(df, dir, from_pickle=False):
    #  # .sample(frac=1) for shuffing
    for pusch_record in df.sample(frac=1).itertuples():
        data_filename = pusch_record.Data_filename
        data_dirname = pusch_record.Data_dirname
        esno_db = pusch_record.Esno_db
        index = pusch_record.Index
        # 1 tx
        if from_pickle:
            b = load_pickle(f'{dir}/{data_dirname}/{data_filename}.b.pkl')
            c = load_pickle(f'{dir}/{data_dirname}/{data_filename}.c.pkl')
            y = load_pickle(f'{dir}/{data_dirname}/{data_filename}.y.pkl')
        else:
            b, c, y = load_hdf5(f'{dir}/{data_dirname}.hdf5', f'{data_filename}')

        c_len = tf.shape(c)[-1]
        b_len = tf.shape(b)[-1]
        b = tf.pad(b, [[0,c_len-b_len]])  # Pad b with zeros to match c
        
        yield index, esno_db, c, y, b, b_len

def preprocessing(index, esno_db, c, y, b, b_len):
    c = tf.transpose(tf.reshape(c, [12,-1,2]), perm=[2,0,1]) # 2 dmrs
    y = tf.concat([tf.math.real(y), tf.math.imag(y)], axis=0)

    return index, esno_db, c, y, b, b_len

dataset_dir = f'../Pusch_data/dataset'
pickles_dir = f'{dataset_dir}/pickle'
hdf5_dir = f'{dataset_dir}/hdf5'
parquet_path = f'{dataset_dir}/parquet/20250304031450018139.parquet'
df = pd.read_parquet(parquet_path, engine="pyarrow")
# df = df[(df['nMCS'] == 9) & (df['nSlot'] == 4)] 

dataset = tf.data.Dataset.from_generator(
            lambda: data_loader(df, hdf5_dir),
            output_types=(tf.int32, tf.float32, tf.float32, tf.complex64, tf.float32, tf.int32))

In [6]:
for n, (index, esno_db, c, y, b, b_len) in enumerate(dataset.map(preprocessing).batch(32)):
    print(index, n, esno_db.shape, c.shape, y.shape, b.shape, b_len.shape)

tf.Tensor([ 5  8  0  1 11  9  2  6  7 10  3  4], shape=(12,), dtype=int32) 0 (12,) (12, 2, 12, 48) (12, 16, 14, 48) (12, 1152) (12,)
